# 三味線譜変換ツール

五線譜（PDF）→ MusicXML（oemer）→ 三味線中間XML の変換を行います。

**実行順序:** セル0（セットアップ）→ セル1（PDF変換）→ セル2（調弦選択）→ セル3（変換）→ セル4（ダウンロード）

In [ ]:
# ============================================================
# セル0: セットアップ（最初に一度だけ実行 / 再実行で最新版に更新）
# ============================================================

import subprocess, sys, os, glob
from google.colab import userdata

REPO_NAME = "kamex120/shamisen-converter"
REPO_DIR  = "shamisen-converter"

def run(cmd, **kwargs):
    r = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if r.stdout: print(r.stdout.strip())
    if r.returncode != 0:
        print("❌ エラー:", r.stderr.strip())
        raise RuntimeError(f"コマンド失敗: {' '.join(cmd)}")
    return r

# リポジトリのクローン or 更新
token = userdata.get("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/{REPO_NAME}.git"
if not os.path.exists(REPO_DIR):
    print("📥 リポジトリをクローン中...")
    run(["git", "clone", repo_url, REPO_DIR])
else:
    print("🔄 最新版に更新中...")
    run(["git", "pull"], cwd=REPO_DIR)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"📁 作業ディレクトリ: {os.getcwd()}")

# 依存ライブラリのインストール
print("\n📦 ライブラリをインストール中...")
subprocess.run(["apt-get", "install", "-y", "poppler-utils", "-q"], capture_output=True)
run([sys.executable, "-m", "pip", "install",
     "oemer", "music21", "pyyaml", "pdf2image",
     "onnxruntime-gpu>=1.18.0",  # GPU対応版（cuDNN 9 / CUDA 12 対応）
     "-q"])

# oemer モデルのダウンロード（不足分のみ）
MODULE_PATH = glob.glob("/usr/local/lib/python*/dist-packages/oemer")[0]
CHECKPOINTS = {
    "unet_big/model.onnx": "https://github.com/BreezeWhite/oemer/releases/download/checkpoints/1st_model.onnx",
    "unet_big/model.h5":   "https://github.com/BreezeWhite/oemer/releases/download/checkpoints/1st_weights.h5",
    "seg_net/model.onnx":  "https://github.com/BreezeWhite/oemer/releases/download/checkpoints/2nd_model.onnx",
    "seg_net/model.h5":    "https://github.com/BreezeWhite/oemer/releases/download/checkpoints/2nd_weights.h5",
}
missing = {k: v for k, v in CHECKPOINTS.items()
           if not os.path.exists(os.path.join(MODULE_PATH, "checkpoints", k))}
if missing:
    print(f"\n📥 oemer モデルをダウンロード中（{len(missing)} ファイル）...")
    for fname, url in missing.items():
        dest = os.path.join(MODULE_PATH, "checkpoints", fname)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        print(f"  {fname} ...")
        subprocess.run(["wget", "-q", "--show-progress", url, "-O", dest], check=True)
    print("✅ モデルダウンロード完了")
else:
    print("\n✅ oemer モデル確認済み")

print("\n✅ セットアップ完了")

In [ ]:
# ============================================================
# セル0-b: Audiveris セットアップ（Audiverisを使う場合のみ実行）
# ============================================================

import subprocess, os, glob, json
from urllib.request import urlopen

AUDIVERIS_DIR = "/content/audiveris_dist"

if os.path.exists(AUDIVERIS_DIR):
    print("✅ Audiveris セットアップ済み")
else:
    # Java と xvfb のインストール
    print("☕ Java をインストール中...")
    subprocess.run(["apt-get", "install", "-y", "default-jre", "xvfb", "-q"], capture_output=True)

    # 最新リリースの ZIP URL を取得
    print("📥 Audiveris の最新バージョンを確認中...")
    with urlopen("https://api.github.com/repos/Audiveris/audiveris/releases/latest") as r:
        release = json.loads(r.read())
    version = release["tag_name"]
    zip_assets = [a for a in release["assets"] if a["name"].endswith(".zip")]
    if not zip_assets:
        raise RuntimeError("Audiveris の ZIP リリースが見つかりません")
    zip_url = zip_assets[0]["browser_download_url"]
    print(f"  バージョン: {version}")

    # ダウンロード & 展開
    print("📥 ダウンロード中（数分かかります）...")
    subprocess.run(["wget", "-q", "--show-progress", zip_url, "-O", "/tmp/audiveris.zip"], check=True)
    subprocess.run(["unzip", "-q", "/tmp/audiveris.zip", "-d", AUDIVERIS_DIR], check=True)
    os.remove("/tmp/audiveris.zip")
    print("✅ Audiveris セットアップ完了")

# 実行ファイルのパスを特定
audiveris_bins = glob.glob(f"{AUDIVERIS_DIR}/**/bin/audiveris", recursive=True)
if not audiveris_bins:
    # JAR ファイルで直接実行するパターンにも対応
    audiveris_jars = glob.glob(f"{AUDIVERIS_DIR}/**/*.jar", recursive=True)
    AUDIVERIS_BIN = None
    AUDIVERIS_JAR = audiveris_jars[0] if audiveris_jars else None
    print(f"  JAR: {AUDIVERIS_JAR}")
else:
    AUDIVERIS_BIN = audiveris_bins[0]
    AUDIVERIS_JAR = None
    subprocess.run(["chmod", "+x", AUDIVERIS_BIN])
    print(f"  実行ファイル: {AUDIVERIS_BIN}")

---
## Step 1: PDF → MusicXML

In [ ]:
# ============================================================
# セル1-b: PDF → MusicXML（oemer または Audiveris）
# ============================================================

import os, subprocess, glob
from pdf2image import convert_from_path
from music21 import converter, stream
from tqdm.notebook import tqdm

OMR_DIR = "_omr_output"
os.makedirs(OMR_DIR, exist_ok=True)
omr_tool = omr_widget.value
print(f"OMRツール: {omr_tool}")

# -------------------------------------------------------
# oemer
# -------------------------------------------------------
if omr_tool == "oemer":
    IMG_DIR = "_pages"
    os.makedirs(IMG_DIR, exist_ok=True)

    print("📃 PDFを画像に変換中...")
    pages = convert_from_path(pdf_path, dpi=300)
    img_paths = []
    for i, page in enumerate(tqdm(pages, desc="PDF→画像")):
        img_path = f"{IMG_DIR}/page_{i+1:02d}.png"
        page.save(img_path, "PNG")
        img_paths.append(img_path)
    print(f"  → {len(img_paths)} ページ")

    print("\n🎵 oemer で楽譜認識中（ONNX Runtime + GPU）...")
    xml_paths = []
    for img_path in tqdm(img_paths, desc="楽譜認識"):
        basename = os.path.splitext(os.path.basename(img_path))[0]
        out_dir = f"{OMR_DIR}/{basename}"
        os.makedirs(out_dir, exist_ok=True)
        tqdm.write(f"  処理中: {img_path}")
        proc = subprocess.run(
            ["oemer", img_path, "-o", out_dir],
            capture_output=True, text=True
        )
        if proc.returncode != 0:
            tqdm.write(f"  ❌ エラー:\n{proc.stderr[-300:]}")
        found = [
            os.path.join(out_dir, f) for f in os.listdir(out_dir)
            if f.endswith(".xml") or f.endswith(".musicxml")
        ]
        if found:
            xml_paths.append(found[0])
            tqdm.write(f"  ✅ → {found[0]}")
        else:
            tqdm.write(f"  ⚠️  出力なし: {os.listdir(out_dir)}")

# -------------------------------------------------------
# Audiveris
# -------------------------------------------------------
else:
    out_dir = f"{OMR_DIR}/audiveris"
    os.makedirs(out_dir, exist_ok=True)
    print("🎵 Audiveris で楽譜認識中...")

    if AUDIVERIS_BIN:
        cmd = ["xvfb-run", AUDIVERIS_BIN,
               "-batch", "-export", "-output", out_dir, "--", pdf_path]
    else:
        cmd = ["xvfb-run", "java", "-jar", AUDIVERIS_JAR,
               "-batch", "-export", "-output", out_dir, "--", pdf_path]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print("❌ Audiveris エラー:", proc.stderr[-300:])

    # .mxl（圧縮MusicXML）を探す
    found_mxl = glob.glob(f"{out_dir}/**/*.mxl", recursive=True)
    found_xml = glob.glob(f"{out_dir}/**/*.xml", recursive=True)
    xml_paths = found_mxl + found_xml
    if xml_paths:
        print(f"  ✅ → {xml_paths[0]}")
    else:
        print(f"  ⚠️  出力なし: {os.listdir(out_dir)}")

# -------------------------------------------------------
# 共通：複数ページのマージ
# -------------------------------------------------------
if not xml_paths:
    raise RuntimeError("MusicXMLが1つも生成されませんでした。上の出力を確認してください。")

if len(xml_paths) == 1:
    musicxml_path = xml_paths[0]
else:
    print("\n📎 複数ページをマージ中...")
    merged = stream.Score()
    for xp in tqdm(xml_paths, desc="マージ"):
        sc = converter.parse(xp)
        for part in sc.parts:
            merged.append(part)
    musicxml_path = f"{OMR_DIR}/merged.xml"
    merged.write("musicxml", fp=musicxml_path)

print(f"\n✅ MusicXML生成完了 → {musicxml_path}")

In [ ]:
# ============================================================
# セル1-a: PDFをアップロード
# ============================================================

from google.colab import files

print("PDFファイルを選択してください")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print(f"\n📄 アップロード: {pdf_path}")

In [ ]:
# ============================================================
# セル1-b: PDF → 画像 → MusicXML（oemer）
# ============================================================

import os, subprocess
from pdf2image import convert_from_path
from music21 import converter, stream
from tqdm.notebook import tqdm

IMG_DIR = "_pages"
OMR_DIR = "_omr_output"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(OMR_DIR, exist_ok=True)

# PDF → PNG（ページごと）
print("📃 PDFを画像に変換中...")
pages = convert_from_path(pdf_path, dpi=300)
img_paths = []
for i, page in enumerate(tqdm(pages, desc="PDF→画像")):
    img_path = f"{IMG_DIR}/page_{i+1:02d}.png"
    page.save(img_path, "PNG")
    img_paths.append(img_path)
print(f"  → {len(img_paths)} ページ")

# ONNX Runtime + GPU で推論（--use-tf は不要）
print("\n🎵 oemer で楽譜認識中（ONNX Runtime + GPU）...")
xml_paths = []
for img_path in tqdm(img_paths, desc="楽譜認識"):
    basename = os.path.splitext(os.path.basename(img_path))[0]
    out_dir = f"{OMR_DIR}/{basename}"
    os.makedirs(out_dir, exist_ok=True)
    tqdm.write(f"  処理中: {img_path}")
    proc = subprocess.run(
        ["oemer", img_path, "-o", out_dir],
        capture_output=True, text=True
    )
    if proc.returncode != 0:
        tqdm.write(f"  ❌ oemer エラー:\n{proc.stderr[-500:]}")
    found = [
        os.path.join(out_dir, f) for f in os.listdir(out_dir)
        if f.endswith(".xml") or f.endswith(".musicxml")
    ]
    if found:
        xml_paths.append(found[0])
        tqdm.write(f"  ✅ → {found[0]}")
    else:
        tqdm.write(f"  ⚠️  出力なし。ディレクトリ内: {os.listdir(out_dir)}")

if not xml_paths:
    raise RuntimeError("MusicXMLが1つも生成されませんでした。上の出力を確認してください。")

# 複数ページの場合はマージ
if len(xml_paths) == 1:
    musicxml_path = xml_paths[0]
else:
    print("\n📎 複数ページをマージ中...")
    merged = stream.Score()
    for xp in tqdm(xml_paths, desc="マージ"):
        sc = converter.parse(xp)
        for part in sc.parts:
            merged.append(part)
    musicxml_path = f"{OMR_DIR}/merged.xml"
    merged.write("musicxml", fp=musicxml_path)

print(f"\n✅ MusicXML生成完了 → {musicxml_path}")

In [ ]:
# ============================================================
# セル1-c: MusicXML を再生して内容を確認
# （初回はサウンドフォントのインストールで少し時間がかかります）
# ============================================================

import subprocess
from IPython.display import Audio, display
from music21 import converter

SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
MIDI_PATH = "/tmp/preview.mid"
WAV_PATH  = "/tmp/preview.wav"

# fluidsynth + サウンドフォントのインストール
if not os.path.exists(SOUNDFONT):
    print("🔧 音声合成エンジンをインストール中（初回のみ）...")
    subprocess.run(["apt-get", "install", "-y", "fluidsynth", "fluid-soundfont-gm", "-q"], capture_output=True)

# MusicXML → MIDI
print("🎵 MusicXML を解析中...")
score = converter.parse(musicxml_path)
score.write("midi", fp=MIDI_PATH)

# MIDI → WAV
print("🔊 音声を生成中...")
result = subprocess.run(
    ["fluidsynth", "-ni", SOUNDFONT, MIDI_PATH, "-F", WAV_PATH, "-r", "44100"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ 音声生成失敗:", result.stderr)
else:
    print("▶️  再生して内容を確認してください:")
    display(Audio(WAV_PATH))

---
## Step 2: 調弦を選択

In [ ]:
# ============================================================
# セル2: 調弦選択
# ============================================================

import ipywidgets as widgets
from IPython.display import display

tuning_widget = widgets.ToggleButtons(
    options=[
        ("本調子", "honchoshi"),
        ("二上り", "niagari"),
        ("三下り", "sansagari"),
    ],
    description="調弦:",
    button_style="info",
)
display(tuning_widget)

---
## Step 3: MusicXML → 三味線中間XML

In [ ]:
# ============================================================
# セル3-a: 変換実行
# ============================================================

from shamisen_converter import (
    convert_musicxml, resolve_out_of_range,
    load_mapping, build_midi_to_position, to_intermediate_xml
)

tuning = tuning_widget.value
print(f"調弦: {tuning_widget.label} ({tuning})")

result = convert_musicxml(
    musicxml_path=musicxml_path,
    mapping_path="shamisen_mapping.yaml",
    tuning=tuning,
)

total = len([n for n in result.notes if n.note_name != "rest"])
out_of_range = len([n for n in result.notes if n.out_of_range])
print(f"\n音符数: {total}")
print(f"音域外: {out_of_range} 件")
if result.warnings:
    print(f"警告数: {len(result.warnings)} 件")

In [ ]:
# ============================================================
# セル3-b: 音域外の処理
# （音域外がない場合はスキップ可）
# ============================================================

mapping = load_mapping("shamisen_mapping.yaml")
midi_map = build_midi_to_position(mapping, tuning)

out_of_range_notes = [n for n in result.notes if n.out_of_range]

if not out_of_range_notes:
    print("✅ 音域外の音符はありません")
else:
    print(f"⚠️  {len(out_of_range_notes)} 件の音域外音符があります")
    print()

    # 一括処理の選択
    policy_widget = widgets.RadioButtons(
        options=[
            ("すべてオクターブ上げて再変換", "up"),
            ("すべてオクターブ下げて再変換", "down"),
            ("すべて休符として扱う", "skip"),
            ("すべて未解決のまま残す", "keep"),
            ("1音ずつ手動で選択", "manual"),
        ],
        description="処理方針:",
        layout=widgets.Layout(width="400px"),
    )
    apply_btn = widgets.Button(description="適用", button_style="success")

    def apply_policy(btn):
        policy = policy_widget.value
        if policy == "manual":
            # interactive=True で1音ずつ処理
            resolve_out_of_range(result, midi_map, interactive=True)
        else:
            for sn in result.notes:
                if not sn.out_of_range:
                    continue
                if policy == "up":
                    new_midi = sn.midi + 12
                    if new_midi in midi_map:
                        s, p = midi_map[new_midi][0]
                        sn.string, sn.position, sn.midi = s, p, new_midi
                        sn.note_name += "(+8va)"
                        sn.out_of_range = False
                        sn.warning = f"オクターブ上げて解決: {sn.note_name}"
                elif policy == "down":
                    new_midi = sn.midi - 12
                    if new_midi in midi_map:
                        s, p = midi_map[new_midi][0]
                        sn.string, sn.position, sn.midi = s, p, new_midi
                        sn.note_name += "(-8va)"
                        sn.out_of_range = False
                        sn.warning = f"オクターブ下げて解決: {sn.note_name}"
                elif policy == "skip":
                    sn.note_name = "rest"
                    sn.out_of_range = False
                    sn.warning = "音域外のためスキップ"
                # keep: そのまま
        print("✅ 音域外処理完了")

    apply_btn.on_click(apply_policy)
    display(policy_widget, apply_btn)

In [ ]:
# ============================================================
# セル3-c: 中間XML出力
# ============================================================

OUTPUT_PATH = "shamisen_output.xml"

xml_str = to_intermediate_xml(result)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(xml_str)

print(f"✅ 変換完了 → {OUTPUT_PATH}")

# 先頭30行をプレビュー
lines = xml_str.splitlines()
print("\n--- プレビュー（先頭30行）---")
print("\n".join(lines[:30]))

---
## Step 4: 中間XMLをダウンロード

In [ ]:
# ============================================================
# セル4: ダウンロード
# ============================================================

from google.colab import files
files.download(OUTPUT_PATH)
print(f"⬇️  {OUTPUT_PATH} をダウンロードしました")

---
## （オプション）MusicXMLを直接アップロードして変換

PDFを使わずMusicXMLファイルから始める場合はこちら。

In [ ]:
# ============================================================
# オプション: MusicXMLを直接アップロード
# ============================================================

from google.colab import files

print("MusicXMLファイル（.xml / .mxl）を選択してください")
uploaded_xml = files.upload()
musicxml_path = list(uploaded_xml.keys())[0]
print(f"\n📄 アップロード: {musicxml_path}")
print("→ セル2（調弦選択）から続けてください")